# Unidad 3 - Sesión 2: Acceso Programático a LLMs

## Ejercicios Prácticos

Este notebook recoge los 5 ejercicios principales + el ejercicio extra de la sesión.
Está pensado para ejecutarse en **Google Colab** usando **OpenRouter** como alternativa gratuita
(aunque incluye también las versiones con la API directa de OpenAI, Gemini y Claude).

### Contenido

1. **Ejercicio 1** — Primera llamada a la API
2. **Ejercicio 2** — Comparativa de APIs
3. **Ejercicio 3** — Chatbot con memoria
4. **Ejercicio 4** — Extracción estructurada (JSON)
5. **Ejercicio 5** — Introducción a LangChain
6. **Ejercicio Extra** — Cliente multi-proveedor unificado

---

### 🔑 Cómo configurar tus API keys en Colab

La forma más segura en Colab es usar **Secrets** (el icono de la llave 🔑 en la barra lateral izquierda):

1. Haz clic en el icono 🔑 de la izquierda en Colab.
2. Añade un secret con nombre `OPENROUTER_API_KEY` y pega tu clave.
3. Activa el toggle "Notebook access" para que este notebook pueda leerla.
4. (Opcional) Haz lo mismo para `OPENAI_API_KEY`, `GOOGLE_API_KEY`, `ANTHROPIC_API_KEY` si quieres usar las APIs directas.

> Si no tienes cuenta, regístrate gratis en [openrouter.ai](https://openrouter.ai) y obtén tu key en [openrouter.ai/keys](https://openrouter.ai/keys).


## 🛠️ Setup inicial

Instalamos todas las dependencias que vamos a necesitar a lo largo del notebook.

In [45]:
# Instalar dependencias (ejecutar una sola vez)
!pip install -q openai python-dotenv google-generativeai anthropic langchain langchain-openai langchain-core

### Cargar las API keys

El siguiente bloque intenta cargar las claves desde los **Secrets de Colab** primero,
y si no están disponibles, las pide por teclado (sin mostrarlas en pantalla).

> ⚠️ **Nunca pegues tu API key directamente en el código.** El archivo `.ipynb` guarda
> todo lo que escribes y es muy fácil que acabe en un repositorio de git, en un screenshot
> o compartido con alguien. Usa siempre Colab Secrets o variables de entorno.

In [46]:
import os

# Intentamos usar los Secrets de Colab
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_key(name, required=False):
    """Obtiene una API key de Colab secrets, variables de entorno o pregunta al usuario."""
    # 1) Colab secrets
    if IN_COLAB:
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return value
        except Exception:
            pass
    # 2) Variable de entorno (útil si usas .env localmente)
    value = os.environ.get(name)
    if value:
        return value
    # 3) Pedir por teclado si es obligatoria
    if required:
        import getpass
        value = getpass.getpass(f"Introduce tu {name}: ")
        os.environ[name] = value
        return value
    return None

# Cargamos la clave de OpenRouter (la usaremos como opción gratuita por defecto)
OPENROUTER_API_KEY = get_key("OPENROUTER_API_KEY", required=True)

# Estas son opcionales: solo hacen falta si quieres probar las APIs directas
OPENAI_API_KEY = get_key("OPENAI_API_KEY")
GOOGLE_API_KEY = get_key("GOOGLE_API_KEY")
ANTHROPIC_API_KEY = get_key("ANTHROPIC_API_KEY")

print("✅ OpenRouter configurado" if OPENROUTER_API_KEY else "❌ Falta OpenRouter")
print("✅ OpenAI configurado" if OPENAI_API_KEY else "⚪ OpenAI no configurado (opcional)")
print("✅ Gemini configurado" if GOOGLE_API_KEY else "⚪ Gemini no configurado (opcional)")
print("✅ Claude configurado" if ANTHROPIC_API_KEY else "⚪ Claude no configurado (opcional)")

✅ OpenRouter configurado
⚪ OpenAI no configurado (opcional)
⚪ Gemini no configurado (opcional)
⚪ Claude no configurado (opcional)


### Descubrir qué modelos gratuitos están disponibles

OpenRouter añade y retira modelos gratuitos con frecuencia (por ejemplo, `google/gemini-2.0-flash-exp:free`
fue retirado en febrero de 2026). Para que el notebook no se rompa, preguntamos directamente a la API
qué modelos `:free` hay disponibles **ahora mismo**.

In [47]:
import requests

def listar_modelos_gratuitos(limite=15):
    """Consulta la API de OpenRouter y devuelve los IDs de modelos :free disponibles."""
    try:
        r = requests.get("https://openrouter.ai/api/v1/models", timeout=10)
        r.raise_for_status()
        modelos = r.json().get("data", [])
        libres = [m["id"] for m in modelos if m["id"].endswith(":free")]
        return libres[:limite]
    except Exception as e:
        print(f"⚠️  No pude consultar la lista de modelos: {e}")
        return []

MODELOS_GRATUITOS = listar_modelos_gratuitos()
print(f"Modelos gratuitos disponibles ({len(MODELOS_GRATUITOS)}):")
for m in MODELOS_GRATUITOS:
    print(f"  • {m}")

# Modelo por defecto: "openrouter/free" es un router que auto-selecciona un modelo gratuito
# y siempre funciona. Es lo que usaremos en la mayoría de los ejercicios.
DEFAULT_MODEL = "openrouter/free"
print(f"\n🎯 Modelo por defecto para los ejercicios: {DEFAULT_MODEL}")

Modelos gratuitos disponibles (15):
  • inclusionai/ling-2.6-flash:free
  • google/gemma-4-26b-a4b-it:free
  • google/gemma-4-31b-it:free
  • nvidia/nemotron-3-super-120b-a12b:free
  • minimax/minimax-m2.5:free
  • arcee-ai/trinity-large-preview:free
  • liquid/lfm-2.5-1.2b-thinking:free
  • liquid/lfm-2.5-1.2b-instruct:free
  • nvidia/nemotron-3-nano-30b-a3b:free
  • nvidia/nemotron-nano-12b-v2-vl:free
  • qwen/qwen3-next-80b-a3b-instruct:free
  • nvidia/nemotron-nano-9b-v2:free
  • openai/gpt-oss-120b:free
  • openai/gpt-oss-20b:free
  • z-ai/glm-4.5-air:free

🎯 Modelo por defecto para los ejercicios: openrouter/free


---

## Ejercicio 1 — Primera llamada a la API

**Objetivo:** hacer la primera llamada a un LLM e interpretar la respuesta (texto, modelo, tokens).

Usamos OpenRouter con un modelo gratuito. Si tienes API key de OpenAI, descomenta las líneas indicadas.

In [48]:
from openai import OpenAI

# Opción A — OpenAI directo (de pago)
# client = OpenAI()
# MODEL = "gpt-4o-mini"

# Opción B — OpenRouter (gratuito) 👇
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = DEFAULT_MODEL  # "openrouter/free" → auto-router

# Primera llamada
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "¿Hola que es una API?"}
    ],
    temperature=0.7
)

# Extraer e imprimir los metadatos
print("Respuesta:", response.choices[0].message.content)
print("\nModelo:", response.model)
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

Respuesta: ¡Hola! Una API (Interfaz de Programación de Aplicaciones) es como un **intermediario** que permite que diferentes programas de software se comuniquen e intercambien información entre sí. Piensa en ella como un camarero en un restaurante: tú (un programa) le pides algo (una función o dato) al camarero (la API), y el camarero se lo lleva al cocinero (otro programa) que lo prepara y lo trae de vuelta a ti.

Aquí te lo explico con más detalle:

**En términos sencillos:**

* **Programas necesitan datos:** Los programas a menudo necesitan información de otros programas para funcionar correctamente. Por ejemplo, una aplicación de clima necesita datos meteorológicos.
* **Las APIs facilitan el acceso:** En lugar de que un programa tenga que "entrar" directamente en el otro programa y acceder a sus datos, la API proporciona una forma estandarizada de solicitar esa información.
* **Un contrato:** La API define qué tipo de solicitudes se pueden hacer y qué tipo de respuestas se pueden e

### Paso 3 — Experimentar con `temperature`

Ejecutamos la misma llamada varias veces con distintos valores de `temperature` y observamos cómo cambian las respuestas.

In [49]:
prompt = "Escribe un eslogan publicitario para una nueva app de fitness."

for temp in [0.0, 0.7, 1.5]:
    print(f"\n{'='*50}")
    print(f"temperature = {temp}")
    print('='*50)
    for i in range(2):  # 2 ejecuciones por temperatura para ver la variabilidad
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
        )
        print(f"[run {i+1}] {r.choices[0].message.content.strip()}")


temperature = 0.0
[run 1] ¡Claro! Aquí tienes algunas opciones de eslóganes para una nueva app de fitness, con diferentes enfoques:

**Cortos y pegadizos:**

*   Tu fitness, a tu ritmo.
*   Muévete. Mejora. Repite.
*   Salud en tu bolsillo.
*   Desbloquea tu mejor versión.
*   Fitness simplificado.

**Enfocados en resultados:**

*   Transforma tu cuerpo, transforma tu vida.
*   Alcanza tus metas, paso a paso.
*   Resultados reales, sin excusas.
*   Empieza hoy, ve la diferencia mañana.
*   Tu viaje fitness, potenciado.

**Enfocados en la comunidad y motivación:**

*   Entrena contigo, inspira a otros.
*   La motivación que necesitas, en tu mano.
*   Únete a la revolución fitness.
*   Tu comunidad fitness, donde sea que estés.
*   Juntos hacia una vida más saludable.

**Más creativos:**

*   Deja que el sudor hable por ti.
*   Redefine tus límites.
*   Tu cuerpo te lo agradecerá.
*   La vida es demasiado corta para no estar en forma.
*   Más que una app, un estilo de vida.

**Para ayud

### 🤔 Preguntas de reflexión

1. ¿Por qué es importante monitorear el consumo de tokens?
2. ¿Qué sucede si envías un prompt muy largo? ¿Cómo afecta a los tokens y al coste?
3. ¿Cuál es la diferencia entre `temperature=0` y `temperature=1.5`?

> Responde aquí tus conclusiones en una celda markdown nueva.

---

## Ejercicio 2 — Comparativa de APIs

**Objetivo:** comparar varios modelos con el mismo prompt midiendo tiempo, tokens y calidad.

Usamos OpenRouter para comparar varios modelos gratuitos con una sola API key. Al final incluyo
también ejemplos (comentados) con las APIs nativas de OpenAI, Gemini y Claude por si quieres probarlos.

In [50]:
import time

prompt = "Explica qué es la recursividad en programación. Incluye un ejemplo en Python."

# Elegimos hasta 3 modelos gratuitos distintos de los que están disponibles HOY.
# Si alguno falla, lo saltamos y seguimos con el siguiente.
if MODELOS_GRATUITOS:
    # Tomamos los primeros 3 IDs únicos
    modelos_a_probar = MODELOS_GRATUITOS[:3]
else:
    # Lista de respaldo por si la API no responde al listado
    modelos_a_probar = [
        "meta-llama/llama-3.3-70b-instruct:free",
        "deepseek/deepseek-r1:free",
        "openai/gpt-oss-120b:free",
    ]

print(f"Modelos a comparar: {modelos_a_probar}\n")

resultados = {}

for modelo in modelos_a_probar:
    print(f"\n{'='*60}\n{modelo}\n{'='*60}")
    try:
        start = time.time()
        response = client.chat.completions.create(
            model=modelo,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        elapsed = time.time() - start

        texto = response.choices[0].message.content
        tokens = response.usage.total_tokens if response.usage else None

        print(texto)
        print(f"\n⏱️  Tiempo: {elapsed:.2f}s")
        print(f"🔢 Tokens: {tokens}")
        print(f"📏 Longitud: {len(texto)} caracteres")

        resultados[modelo] = {
            "tiempo": elapsed,
            "tokens": tokens,
            "longitud": len(texto),
            "respuesta": texto,
        }
    except Exception as e:
        print(f"❌ Error: {e}")
        resultados[modelo] = None

Modelos a comparar: ['inclusionai/ling-2.6-flash:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free']


inclusionai/ling-2.6-flash:free
La **recursividad** en programación es una técnica donde una función se llama a sí misma directamente o indirectamente para resolver un problema. En lugar de usar bucles (como `for` o `while`), la recursividad divide un problema en subproblemas más pequeños y los resuelve llamándose repetidamente hasta llegar a un caso base que detiene la recursión.

### Elementos clave de la recursividad:
1. **Caso base**: Condición que detiene la recursión para evitar bucles infinitos.
2. **Llamada recursiva**: La función se invoca a sí misma con un problema más simple o reducido.
3. **Progreso hacia el caso base**: Cada llamada debe acercarse más al caso base.

---

### Ejemplo en Python: Calcular el factorial de un número

El factorial de un número \( n \) (escrito como \( n! \)) es el producto de todos los enteros positivos desde 1 hasta \( n \).

### Tabla resumen

Imprimimos una tabla comparativa con los resultados.

In [51]:
print(f"{'Modelo':<50}{'Tiempo (s)':<12}{'Tokens':<10}{'Caracteres':<12}")
print("-" * 84)
for modelo, r in resultados.items():
    if r:
        print(f"{modelo:<50}{r['tiempo']:<12.2f}{str(r['tokens']):<10}{r['longitud']:<12}")
    else:
        print(f"{modelo:<50}{'-':<12}{'-':<10}{'-':<12}")

Modelo                                            Tiempo (s)  Tokens    Caracteres  
------------------------------------------------------------------------------------
inclusionai/ling-2.6-flash:free                   4.19        747       2112        
google/gemma-4-26b-a4b-it:free                    18.56       882       3073        
google/gemma-4-31b-it:free                        23.82       826       2862        


### (Opcional) Mismo prompt con las APIs nativas

Solo ejecuta estas celdas si tienes configuradas las API keys correspondientes.

In [52]:
# --- OpenAI nativo ---
if OPENAI_API_KEY:
    from openai import OpenAI as OpenAINative
    cli = OpenAINative(api_key=OPENAI_API_KEY)
    start = time.time()
    r = cli.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    print("=== OpenAI gpt-4o-mini ===")
    print(r.choices[0].message.content)
    print(f"\nTiempo: {time.time()-start:.2f}s | Tokens: {r.usage.total_tokens}")
else:
    print("⚪ Saltando OpenAI (no hay API key)")

⚪ Saltando OpenAI (no hay API key)


In [53]:
# --- Google Gemini nativo ---
if GOOGLE_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel("gemini-1.5-flash")
    start = time.time()
    r = model.generate_content(prompt)
    print("=== Google Gemini 1.5 Flash ===")
    print(r.text)
    print(f"\nTiempo: {time.time()-start:.2f}s")
    print(f"Tokens prompt: {r.usage_metadata.prompt_token_count}")
    print(f"Tokens resp.:  {r.usage_metadata.candidates_token_count}")
else:
    print("⚪ Saltando Gemini (no hay API key)")

⚪ Saltando Gemini (no hay API key)


In [54]:
# --- Anthropic Claude nativo ---
if ANTHROPIC_API_KEY:
    import anthropic
    cli = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    start = time.time()
    msg = cli.messages.create(
        model="claude-3-5-haiku-latest",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    print("=== Anthropic Claude 3.5 Haiku ===")
    print(msg.content[0].text)
    print(f"\nTiempo: {time.time()-start:.2f}s")
    print(f"Tokens entrada: {msg.usage.input_tokens}")
    print(f"Tokens salida:  {msg.usage.output_tokens}")
else:
    print("⚪ Saltando Claude (no hay API key)")

⚪ Saltando Claude (no hay API key)


### 🤔 Preguntas de reflexión

1. ¿Cuál de los modelos dio la mejor respuesta? ¿Por qué?
2. ¿Cuál fue el más rápido? ¿La velocidad importa siempre?
3. ¿En qué escenarios elegirías cada proveedor?
4. ¿Notas diferencias en cómo estructura la respuesta cada modelo?

---

## Ejercicio 3 — Chatbot con memoria

**Objetivo:** construir un chatbot que mantenga el contexto de la conversación recortando el historial cuando se pase de un máximo de mensajes.

La API de LLMs es *stateless*: debemos enviar todo el historial en cada llamada.

In [55]:
# Cliente y modelo (misma configuración que antes)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = DEFAULT_MODEL  # "openrouter/free" → auto-router

SYSTEM_PROMPT = (
    "Eres un tutor de Python amigable y paciente. "
    "Te presentas como 'PyTutor'. "
    "Das explicaciones claras, concisas y siempre acompañadas de un ejemplo de código. "
    "Adaptas tu nivel al del alumno y relacionas conceptos nuevos con los ya tratados."
)

MAX_MESSAGES = 10  # Máximo de mensajes de conversación (sin contar el system prompt)


def create_initial_messages():
    """Lista inicial con el system prompt."""
    return [{"role": "system", "content": SYSTEM_PROMPT}]


def trim_history(messages):
    """Recorta el historial dejando system + últimos MAX_MESSAGES mensajes."""
    if len(messages) - 1 > MAX_MESSAGES:
        # Conservar el system prompt (índice 0) y los últimos MAX_MESSAGES
        messages = [messages[0]] + messages[-MAX_MESSAGES:]
    return messages


def get_response(messages):
    """Llama a la API y devuelve el objeto response completo."""
    return client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
    )

### Versión para notebook (sin `input()` bloqueante)

En Colab es más cómodo tener una función `send()` que se puede llamar varias veces desde celdas distintas,
en vez de un bucle `while` con `input()` (que también funciona, pero resulta pesado de probar).

In [56]:
# Historial global de la conversación
messages = create_initial_messages()

def send(user_input, verbose=True):
    """Envía un mensaje al tutor y actualiza el historial."""
    global messages
    messages.append({"role": "user", "content": user_input})
    messages = trim_history(messages)

    response = get_response(messages)
    assistant_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})

    if verbose:
        print(f"🧑 Tú: {user_input}")
        print(f"\n🤖 PyTutor: {assistant_message}")
        usage = response.usage
        if usage:
            print(
                f"\n   [tokens → prompt: {usage.prompt_tokens}, "
                f"respuesta: {usage.completion_tokens}, total: {usage.total_tokens}]"
            )
        print(f"   [mensajes en historial: {len(messages) - 1}]\n")
    return assistant_message


def reset():
    """Reinicia la conversación."""
    global messages
    messages = create_initial_messages()
    print("🔄 Conversación reiniciada.")

### Paso 2 — Prueba de memoria

Secuencia de prueba para verificar que el bot recuerda el contexto.

In [57]:
reset()
send("¿Qué son las variables en Python?")

🔄 Conversación reiniciada.
🧑 Tú: ¿Qué son las variables en Python?

🤖 PyTutor: ¡Hola! Soy PyTutor, encantado de ayudarte con Python.

Las variables en Python son como contenedores que almacenan información o datos. Piensa en ellas como etiquetas que puedes asignar a valores para poder referirte a ellos más tarde en tu código.

Características importantes de las variables en Python:
- No necesitas declarar su tipo (Python lo infiere automáticamente)
- Se crean cuando les asignas un valor por primera vez
- Pueden cambiar de tipo durante la ejecución del programa
- Los nombres deben comenzar con letra o guion bajo

Aquí te muestro un ejemplo:

```python
# Asignando valores a variables
nombre = "Ana"          # Variable de tipo cadena de texto
edad = 25               # Variable de tipo número entero
altura = 1.68           # Variable de tipo número decimal
es_estudiante = True    # Variable de tipo booleano

# Mostrando los valores
print(f"Nombre: {nombre}")
print(f"Edad: {edad}")
print(f"

'¡Hola! Soy PyTutor, encantado de ayudarte con Python.\n\nLas variables en Python son como contenedores que almacenan información o datos. Piensa en ellas como etiquetas que puedes asignar a valores para poder referirte a ellos más tarde en tu código.\n\nCaracterísticas importantes de las variables en Python:\n- No necesitas declarar su tipo (Python lo infiere automáticamente)\n- Se crean cuando les asignas un valor por primera vez\n- Pueden cambiar de tipo durante la ejecución del programa\n- Los nombres deben comenzar con letra o guion bajo\n\nAquí te muestro un ejemplo:\n\n```python\n# Asignando valores a variables\nnombre = "Ana"          # Variable de tipo cadena de texto\nedad = 25               # Variable de tipo número entero\naltura = 1.68           # Variable de tipo número decimal\nes_estudiante = True    # Variable de tipo booleano\n\n# Mostrando los valores\nprint(f"Nombre: {nombre}")\nprint(f"Edad: {edad}")\nprint(f"Altura: {altura} m")\nprint(f"¿Es estudiante?: {es_estud

In [58]:
send("Dame un ejemplo de lo anterior")

🧑 Tú: Dame un ejemplo de lo anterior

🤖 PyTutor: ¡Claro! Vamos a explorar un ejemplo práctico para que quede más claro.

Imagina que estás creando un programa que necesite almacenar información sobre estudiantes. Puedes usar variables para guardar datos como nombres, edades, etc.

### Ejemplo: Gestión de estudiantes

```python
# Crear variables para almacenar información de estudiantes
nombre = "Luis"        # Nombre del estudiante
edad = 15              # Edad del estudiante
matricula = 101       # Identificador único del estudiante
estudios = "Matemáticas"  # Estudios que cursa

# Mostrar la información
print(f"Nombre: {nombre}")
print(f"Edad: {edad}")
print(f"Matrícula: {matricula}")
print(f"Estudios: {estudios}")
```

### ¿Qué pasamos aquí?
- **Variables**: Al asignar valores, creas contenedores para almacenar datos.
- **Tipos de datos**: Python puede manejar varios tipos (como texto, números, booleanos).
- **Modificarse**: Puedes cambiar el valor de una variable más adelante.

###

'¡Claro! Vamos a explorar un ejemplo práctico para que quede más claro.\n\nImagina que estás creando un programa que necesite almacenar información sobre estudiantes. Puedes usar variables para guardar datos como nombres, edades, etc.\n\n### Ejemplo: Gestión de estudiantes\n\n```python\n# Crear variables para almacenar información de estudiantes\nnombre = "Luis"        # Nombre del estudiante\nedad = 15              # Edad del estudiante\nmatricula = 101       # Identificador único del estudiante\nestudios = "Matemáticas"  # Estudios que cursa\n\n# Mostrar la información\nprint(f"Nombre: {nombre}")\nprint(f"Edad: {edad}")\nprint(f"Matrícula: {matricula}")\nprint(f"Estudios: {estudios}")\n```\n\n### ¿Qué pasamos aquí?\n- **Variables**: Al asignar valores, creas contenedores para almacenar datos.\n- **Tipos de datos**: Python puede manejar varios tipos (como texto, números, booleanos).\n- **Modificarse**: Puedes cambiar el valor de una variable más adelante.\n\n### Resultado:\n```\nNombr

In [59]:
send("Ahora muéstrame cómo usar listas")

🧑 Tú: Ahora muéstrame cómo usar listas

🤖 PyTutor: ¡Excelente pregunta! Las listas son una de las estructuras de datos más útiles en Python. Son como contenedores que pueden almacenar múltiples valores en un solo lugar, y esos valores pueden ser de diferentes tipos.

### ¿Qué son las listas?
Una lista es una colección ordenada de elementos, donde cada elemento tiene un índice (posición) que comienza desde 0.

### Ejemplo: Crear y usar una lista

```python
# Crear una lista de nombres de estudiantes
estudiantes = ["Ana", "Luis", "María", "Carlos"]

# Mostrar la lista completa
print(estudiantes)

# Acceder a elementos específicos usando su índice
print(f"Primer estudiante: {estudiantes[0]}")  # Ana
print(f"Segundo estudiante: {estudiantes[1]}")  # Luis

# Añadir un nuevo estudiante a la lista
estudiantes.append("Pedro")
print(estudiantes)

# Modificar un elemento existente
estudiantes[2] = "Isabel"
print(estudiantes)

# Eliminar un estudiante
estudiantes.remove("Carlos")
print(estudiante

'¡Excelente pregunta! Las listas son una de las estructuras de datos más útiles en Python. Son como contenedores que pueden almacenar múltiples valores en un solo lugar, y esos valores pueden ser de diferentes tipos.\n\n### ¿Qué son las listas?\nUna lista es una colección ordenada de elementos, donde cada elemento tiene un índice (posición) que comienza desde 0.\n\n### Ejemplo: Crear y usar una lista\n\n```python\n# Crear una lista de nombres de estudiantes\nestudiantes = ["Ana", "Luis", "María", "Carlos"]\n\n# Mostrar la lista completa\nprint(estudiantes)\n\n# Acceder a elementos específicos usando su índice\nprint(f"Primer estudiante: {estudiantes[0]}")  # Ana\nprint(f"Segundo estudiante: {estudiantes[1]}")  # Luis\n\n# Añadir un nuevo estudiante a la lista\nestudiantes.append("Pedro")\nprint(estudiantes)\n\n# Modificar un elemento existente\nestudiantes[2] = "Isabel"\nprint(estudiantes)\n\n# Eliminar un estudiante\nestudiantes.remove("Carlos")\nprint(estudiantes)\n\n# Obtener la lon

In [60]:
send("¿Cuál es la diferencia entre lo primero que me explicaste y esto?")

🧑 Tú: ¿Cuál es la diferencia entre lo primero que me explicaste y esto?

🤖 PyTutor: ¡Excelente pregunta! Es importante entender las diferencias claramente.

La diferencia principal es que **en la primera explicación hablamos de variables individuales** (cada una almacena un solo valor), mientras que **en el ejemplo de listas usamos una sola variable que contiene múltiples valores** organizados en una estructura.

### Comparación clara:

**Variables individuales (primera explicación):**
```python
nombre = "Ana"
edad = 25
altura = 1.68
```
- Cada variable almacena un solo valor
- Necesitamos crear una variable por cada dato
- Accedemos a cada variable por su nombre único

**Listas (ejemplo actual):**
```python
estudiantes = ["Ana", "Luis", "María", "Carlos"]
```
- Una sola variable almacena múltiples valores
- Todos los valores están relacionados entre sí
- Accedemos a cada elemento por su posición (índice)

### ¿Por qué usar listas entonces?

Las listas son ideales cuando necesitas:
- *

'¡Excelente pregunta! Es importante entender las diferencias claramente.\n\nLa diferencia principal es que **en la primera explicación hablamos de variables individuales** (cada una almacena un solo valor), mientras que **en el ejemplo de listas usamos una sola variable que contiene múltiples valores** organizados en una estructura.\n\n### Comparación clara:\n\n**Variables individuales (primera explicación):**\n```python\nnombre = "Ana"\nedad = 25\naltura = 1.68\n```\n- Cada variable almacena un solo valor\n- Necesitamos crear una variable por cada dato\n- Accedemos a cada variable por su nombre único\n\n**Listas (ejemplo actual):**\n```python\nestudiantes = ["Ana", "Luis", "María", "Carlos"]\n```\n- Una sola variable almacena múltiples valores\n- Todos los valores están relacionados entre sí\n- Accedemos a cada elemento por su posición (índice)\n\n### ¿Por qué usar listas entonces?\n\nLas listas son ideales cuando necesitas:\n- **Grupos de datos relacionados** (como una lista de estud

### Paso 3 — Probar el límite de historial

Bajamos `MAX_MESSAGES` a 4 y observamos qué pasa cuando la conversación se alarga.

In [61]:
MAX_MESSAGES = 4
reset()

preguntas = [
    "¿Qué son las funciones en Python?",
    "¿Cómo se pasan argumentos por defecto?",
    "¿Y los args/kwargs?",
    "Dame un ejemplo con decoradores",
    "¿Recuerdas cuál fue mi primera pregunta?",  # probablemente ya no la recuerde
]

for p in preguntas:
    send(p)
    print("-" * 60)

🔄 Conversación reiniciada.
🧑 Tú: ¿Qué son las funciones en Python?

🤖 PyTutor: ¡Claro! Las **funciones** en Python son bloques de código que realizan una tarea específica. Son como "pequeños programas" dentro de tu programa principal. Las funciones te permiten **reutilizar código**, hacerlo más organizado y fácil de entender. 

### ¿Cómo funcionan?
1. **Definir una función**: Usas la palabra clave `def` seguida del nombre de la función, paréntesis y dos puntos.  
2. **Ejecutar la función**: Llamas a la función con sus paréntesis, y se ejecuta el código dentro de su bloque.  

### Ejemplo básico:
```python
def saludar():
    print("¡Hola, mundo!")

# Llamar a la función
saludar()
```
**Salida:**  
```
¡Hola, mundo!
```

### ¿Por qué usarlas?
- **Reutilidad**: Puedes llamar a la función tantas veces como quieras.  
- **Modularidad**: Divides tu código en partes más pequeñas y manejables.  
- **Claridad**: Hace que tu código sea más fácil de leer y mantener.  

¿Quieres que te muestre un 

### Paso 4 (Bonus) — Resumen del historial antiguo

En vez de tirar los mensajes antiguos, los resumimos y guardamos el resumen como contexto.

In [62]:
def summarize_and_trim(messages, max_messages=MAX_MESSAGES):
    """
    Si el historial pasa de max_messages, genera un resumen de los antiguos
    y lo inserta como un mensaje system adicional justo después del system prompt.
    """
    if len(messages) - 1 <= max_messages:
        return messages

    # El system prompt original es messages[0]
    system_msg = messages[0]
    # Los mensajes que vamos a descartar (todos menos los últimos max_messages)
    old_messages = messages[1:-max_messages]
    # Los que conservamos tal cual
    recent_messages = messages[-max_messages:]

    # Construimos un prompt para resumir
    conversation_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in old_messages
    )
    summary_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Resume de forma muy breve (2-3 oraciones) los puntos clave "
                    "de la siguiente conversación entre un alumno y un tutor de Python. "
                    "Conserva los conceptos tratados, no cites los mensajes literalmente."
                ),
            },
            {"role": "user", "content": conversation_text},
        ],
        temperature=0.3,
    )
    resumen = summary_response.choices[0].message.content.strip()

    resumen_msg = {
        "role": "system",
        "content": f"[Resumen de la conversación previa]: {resumen}",
    }

    return [system_msg, resumen_msg] + recent_messages


# Probamos la versión con resumen
messages = create_initial_messages()
for p in preguntas:
    messages.append({"role": "user", "content": p})
    messages = summarize_and_trim(messages, max_messages=4)
    r = get_response(messages)
    ans = r.choices[0].message.content
    messages.append({"role": "assistant", "content": ans})
    print(f"🧑 {p}\n🤖 {ans}\n{'-'*60}")

print("\n🗂️  Historial final:")
for m in messages:
    preview = m['content'][:80].replace("\n", " ")
    print(f"  [{m['role']}] {preview}...")

🧑 ¿Qué son las funciones en Python?
🤖 ¡Claro! Las **funciones** son bloques de código que realizan una tarea específica, como calcular un valor o organizar datos. Son útiles para evitar repetir código y hacerlo más legible.  

**Ejemplo sencillo:**  
```python
# Función para calcular la suma de dos números
def sumar(a, b):
    return a + b

resultado = sumar(3, 5)  # Llama a la función
print(resultado)    # Imprime 8
```

**¿Por qué usar funciones?**  
- **Reutilización:** Si necesitas sumar dos veces, solo escribas la función una vez.  
- **Claridad:** Divide tareas complejas en partes pequeñas.  

¿Quieres practicar con funciones personalizadas o resolver un problema específico? ¡Estoy aquí para ayudarte! 😊
------------------------------------------------------------
🧑 ¿Cómo se pasan argumentos por defecto?
🤖 ¡Excelente pregunta! Los argumentos por defecto en Python te permiten definir valores predeterminados para los parámetros de una función. Si al llamar la función no se proporcio

---

## Ejercicio 4 — Extracción estructurada (JSON)

**Objetivo:** convertir texto libre en datos estructurados (JSON) validando el formato y con reintentos.

In [63]:
import os
import json
import time
from openai import OpenAI, RateLimitError, APIError

# --- Cliente ---
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = "openrouter/free"

SYSTEM_PROMPT = """Eres un sistema de extracción de información. Tu tarea es extraer datos
estructurados de textos no estructurados y devolver ÚNICAMENTE un JSON válido.

Reglas:
- Responde SOLO con el JSON, sin texto adicional, sin bloques de código markdown.
- Si un campo no se encuentra en el texto, usa el valor "No especificado".
- Los valores numéricos deben ser números, no strings.
- Las fechas deben estar en formato YYYY-MM-DD cuando sea posible.
"""


def extract_json(text, schema_description, max_retries=3, max_rate_waits=2):
    """
    Extrae datos estructurados de un texto libre con:
      - reintentos si el JSON viene mal formado
      - espera inteligente ante RateLimitError (429)
    """
    user_prompt = f"""Extrae la información del siguiente texto y devuelve un JSON
con este esquema:
{schema_description}

Texto:
\"\"\"
{text}
\"\"\"
"""

    rate_waits = 0

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0,
            )
            content = response.choices[0].message.content.strip()

            # Limpia posibles bloques ```json ... ```
            if content.startswith("```"):
                content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()

            result = json.loads(content)
            print(f"  ✅ Extracción exitosa en intento {attempt + 1}")
            return result

        except RateLimitError as e:
            # Intentamos leer cuándo se resetea el rate limit
            reset_ms = None
            try:
                reset_ms = int(e.response.headers.get("X-RateLimit-Reset", 0))
            except Exception:
                pass

            if reset_ms:
                wait_s = max(0, reset_ms / 1000 - time.time())
            else:
                wait_s = 30  # fallback conservador

            # Si el reset es muy lejano (tier diario agotado), no tiene sentido esperar
            if wait_s > 300:  # > 5 minutos
                print(f"  🛑 Rate limit diario agotado. Reset en {wait_s/3600:.1f} h.")
                print(f"     Opciones: esperar al reset (medianoche UTC) o añadir 10 USD")
                print(f"     de crédito en OpenRouter para obtener 1000 req/día gratis.")
                return None

            rate_waits += 1
            if rate_waits > max_rate_waits:
                print(f"  ❌ Demasiados rate limits seguidos, abandono este texto.")
                return None

            print(f"  ⏳ Rate limit, esperando {wait_s:.1f}s antes de reintentar…")
            time.sleep(wait_s + 1)
            # Reintentamos sin consumir attempt
            continue

        except json.JSONDecodeError as e:
            print(f"  ⚠️  Intento {attempt + 1}/{max_retries} - JSON inválido: {e}")
            if attempt == max_retries - 1:
                print(f"  ❌ Se agotaron los reintentos.")
                print(f"  Contenido bruto recibido:\n{content}")
                return None

        except APIError as e:
            print(f"  ❌ Error de la API: {e}")
            return None

    return None


### Textos de entrada

In [64]:

texto_empleo = """
¡Únete a nuestro equipo! Buscamos Desarrollador Senior Python para nuestra
oficina en Madrid. Ofrecemos salario de 45.000-55.000€ brutos anuales,
teletrabajo 3 días por semana y seguro médico privado. Requisitos: 5 años
de experiencia, conocimientos en Django y PostgreSQL. Incorporación
inmediata.
Enviar CV a empleo@techcorp.es antes del 15 de marzo de 2025.
"""

texto_resena = """
Compré el portátil UltraBook X15 hace 2 semanas. La pantalla de 15 pulgadas
es espectacular y la batería dura unas 10 horas reales. Sin embargo, el
teclado es un poco incómodo para escribir largo rato y se calienta bastante
con tareas pesadas. Por el precio de 1.299€ creo que está bien, pero no es
perfecto. Le doy un 7 de 10. Lo compré en Amazon el 20 de enero de 2025.
"""

texto_noticia = """
La empresa española de inteligencia artificial, NovaTech, anunció hoy una
ronda de financiación Serie B por valor de 30 millones de euros, liderada
por el fondo Sequoia Capital con participación de Telefónica Ventures.
La compañía, fundada en 2021 por María García y Carlos López, planea usar
los fondos para expandirse a Latinoamérica y contratar a 50 ingenieros
antes de fin de año. NovaTech ha desarrollado un modelo de lenguaje
especializado en el sector legal.
"""

textos = {
    "Oferta de empleo": (
        texto_empleo,
        "puesto, empresa, ubicacion, salario_min, salario_max, modalidad, "
        "requisitos (lista), beneficios (lista), contacto, fecha_limite",
    ),
    "Reseña de producto": (
        texto_resena,
        "producto, puntuacion, puntuacion_maxima, precio, aspectos_positivos (lista), "
        "aspectos_negativos (lista), fecha_compra, tienda, recomendacion_general",
    ),
    "Noticia": (
        texto_noticia,
        "empresa, tipo_evento, monto, moneda, inversores (lista), fundadores (lista), "
        "año_fundacion, sector, planes (lista)",
    ),
}

# 👉 Cuando vuelvas a tener cupo, puedes reejecutar solo lo que falta cambiando
#    esta lista, por ejemplo: textos_a_procesar = ["Noticia"]
textos_a_procesar = list(textos.keys())   # los 3
# textos_a_procesar = ["Noticia"]         # solo el que falló

PAUSA_ENTRE_TEXTOS = 2  # segundos, para no hacer ráfagas

for nombre in textos_a_procesar:
    texto, esquema = textos[nombre]
    print(f"\n{'='*60}\nProcesando: {nombre}\n{'='*60}")
    resultado = extract_json(texto, esquema)
    if resultado:
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    time.sleep(PAUSA_ENTRE_TEXTOS)


Procesando: Oferta de empleo
  ⚠️  Intento 1/3 - JSON inválido: Extra data: line 1 column 9 (char 8)
  ✅ Extracción exitosa en intento 2
{
  "puesto": "Desarrollador Senior Python",
  "empresa": "No especificado",
  "ubicacion": "Madrid",
  "salario_min": 45000,
  "salario_max": 55000,
  "modalidad": "Híbrido (teletrabajo 3 días por semana)",
  "requisitos": [
    "5 años de experiencia",
    "Conocimientos en Django",
    "Conocimientos en PostgreSQL"
  ],
  "beneficios": [
    "Seguro médico privado"
  ],
  "contacto": "empleo@techcorp.es",
  "fecha_limite": "2025-03-15"
}

Procesando: Reseña de producto
  ❌ Error de la API: Error code: 402 - {'error': {'message': 'Provider returned error', 'code': 402, 'metadata': {'raw': '{"error":"API key USD spend limit exceeded. Your account may still have USD balance, but this API key has reached its configured USD spending limit."}', 'provider_name': 'Venice', 'is_byok': False}}, 'user_id': 'user_3CiqSATQ0bmBGlPz1fRksjAlaSK'}

Procesando: Not

### 🤔 Análisis

1. ¿En cuántos intentos logró generar JSON válido para cada texto?
2. ¿Hubo campos con `"No especificado"`? ¿Era correcto?
3. ¿Los valores numéricos se devolvieron como números o como strings?
4. ¿Qué pasaría si el texto estuviera en otro idioma?

---

## Ejercicio 5 — Introducción a LangChain

**Objetivo:** replicar la extracción del ejercicio 4 usando LangChain y el patrón `prompt | model | parser`.

In [65]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print(f"LangChain versión: {langchain.__version__}")

LangChain versión: 1.2.15


In [66]:
# 1. Modelo — usamos OpenRouter a través del wrapper de OpenAI
model = ChatOpenAI(
    model="openrouter/free",  # auto-router que elige un modelo gratuito disponible
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# Si tuvieras API key de OpenAI directamente:
# model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Template del prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Eres un sistema de extracción de información. Tu tarea es extraer datos
estructurados de textos no estructurados y devolver ÚNICAMENTE un JSON válido.

Reglas:
- Responde SOLO con el JSON, sin texto adicional, sin bloques de código markdown.
- Si un campo no se encuentra en el texto, usa el valor "No especificado".
- Los valores numéricos deben ser números, no strings.
- Las fechas deben estar en formato YYYY-MM-DD cuando sea posible.""",
    ),
    (
        "user",
        """Extrae la información del siguiente texto y devuelve un JSON
con este esquema: {schema}

Texto:
\"\"\"
{text}
\"\"\"
""",
    ),
])

# 3. Parser de salida
output_parser = StrOutputParser()

# 4. Chain con el operador pipe
chain = prompt | model | output_parser

In [67]:
# Probamos con un solo texto
result = chain.invoke({
    "text": texto_empleo,
    "schema": "puesto, empresa, ubicacion, salario_min, salario_max, modalidad, "
              "requisitos (lista), beneficios (lista), contacto, fecha_limite",
})
print(result)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1776879480000'}, 'provider_name': None}}, 'user_id': 'user_3CiqSATQ0bmBGlPz1fRksjAlaSK'}

### Paso 3 — Procesar los tres textos

In [ ]:
textos_lc = {
    "Oferta de empleo": {
        "text": texto_empleo,
        "schema": "puesto, empresa, ubicacion, salario_min, salario_max, modalidad, "
                  "requisitos (lista), beneficios (lista), contacto, fecha_limite",
    },
    "Reseña de producto": {
        "text": texto_resena,
        "schema": "producto, puntuacion, puntuacion_maxima, precio, aspectos_positivos (lista), "
                  "aspectos_negativos (lista), fecha_compra, tienda, recomendacion_general",
    },
    "Noticia": {
        "text": texto_noticia,
        "schema": "empresa, tipo_evento, monto, moneda, inversores (lista), fundadores (lista), "
                  "año_fundacion, sector, planes (lista)",
    },
}

for nombre, inputs in textos_lc.items():
    print(f"\n{'='*60}\nProcesando: {nombre}\n{'='*60}")
    raw = chain.invoke(inputs)

    # Limpieza de posible markdown
    if raw.strip().startswith("```"):
        raw = raw.strip().split("\n", 1)[1].rsplit("```", 1)[0].strip()

    try:
        parsed = json.loads(raw)
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
    except json.JSONDecodeError as e:
        print(f"❌ Error al parsear JSON: {e}")
        print(raw)

### 🤔 Comparación API nativa vs LangChain

| Aspecto | API nativa (Ej. 4) | LangChain (Ej. 5) |
|---|---|---|
| Líneas de código | Más (reintentos, parseo manual) | Menos (abstracciones preparadas) |
| Cambiar de modelo | Requiere cambiar cliente + schema | Cambiar una línea |
| Reutilización de prompts | Strings sueltos | `ChatPromptTemplate` |
| Composición | Funciones encadenadas a mano | Operador `\|` |
| Control fino | Total | Más abstracto |

Preguntas:
1. ¿Cuántas líneas te ahorraste con LangChain?
2. ¿Qué beneficios ves en el patrón `prompt | model | parser`?
3. ¿En qué casos preferirías la API nativa?
4. ¿Cómo cambiarías la chain para usar Claude en lugar de OpenRouter?

---

## Ejercicio Extra — Cliente multi-proveedor unificado

**Objetivo:** una sola clase `LLMClient` que abstraiga las diferencias entre OpenAI, Gemini, Claude y OpenRouter.

Formato unificado de mensajes (estilo OpenAI):

```python
messages = [
    {"role": "system", "content": "Eres un asistente útil."},
    {"role": "user",   "content": "Hola, ¿cómo estás?"}
]
```

La clase se encarga internamente de adaptar ese formato a lo que espera cada SDK.

In [ ]:
class LLMClient:
    """Cliente unificado para múltiples proveedores de LLMs."""

    DEFAULT_MODELS = {
        "openai":     "gpt-4o-mini",
        "gemini":     "gemini-1.5-flash",
        "claude":     "claude-3-5-haiku-latest",
        "openrouter": "openrouter/free",
    }

    def __init__(self, provider: str, model: str = None):
        if provider not in self.DEFAULT_MODELS:
            raise ValueError(
                f"Proveedor no soportado: {provider}. "
                f"Usa: {list(self.DEFAULT_MODELS.keys())}"
            )
        self.provider = provider
        self.model = model or self.DEFAULT_MODELS[provider]
        self._init_client()

    # ------------------------------------------------------------
    # Inicialización del SDK concreto
    # ------------------------------------------------------------
    def _init_client(self):
        if self.provider == "openai":
            from openai import OpenAI
            self._client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        elif self.provider == "openrouter":
            from openai import OpenAI
            self._client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=os.getenv("OPENROUTER_API_KEY"),
            )

        elif self.provider == "gemini":
            import google.generativeai as genai
            genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
            self._genai = genai  # guardamos el módulo para crear modelos con system_instruction

        elif self.provider == "claude":
            import anthropic
            self._client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    # ------------------------------------------------------------
    # Adaptación del formato de mensajes a cada API
    # ------------------------------------------------------------
    @staticmethod
    def _split_system(messages):
        """Devuelve (system_text, messages_sin_system)."""
        system_text = ""
        rest = []
        for m in messages:
            if m["role"] == "system":
                system_text += ("\n" if system_text else "") + m["content"]
            else:
                rest.append(m)
        return system_text, rest

    # ------------------------------------------------------------
    # Chat síncrono
    # ------------------------------------------------------------
    def chat(self, messages, **kwargs):
        temperature = kwargs.get("temperature", 0.7)
        max_tokens = kwargs.get("max_tokens", 1024)

        if self.provider in ("openai", "openrouter"):
            response = self._client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content

        elif self.provider == "gemini":
            system_text, rest = self._split_system(messages)
            model = self._genai.GenerativeModel(
                self.model,
                system_instruction=system_text or None,
            )
            # Gemini alterna user/model; pasamos el último mensaje del usuario
            # y el resto como historial.
            history = []
            for m in rest[:-1]:
                role = "user" if m["role"] == "user" else "model"
                history.append({"role": role, "parts": [m["content"]]})
            chat = model.start_chat(history=history)
            last = rest[-1]["content"]
            response = chat.send_message(
                last,
                generation_config={"temperature": temperature, "max_output_tokens": max_tokens},
            )
            return response.text

        elif self.provider == "claude":
            system_text, rest = self._split_system(messages)
            response = self._client.messages.create(
                model=self.model,
                max_tokens=max_tokens,
                temperature=temperature,
                system=system_text or None,
                messages=rest,
            )
            return response.content[0].text

    # ------------------------------------------------------------
    # Streaming
    # ------------------------------------------------------------
    def stream(self, messages, **kwargs):
        temperature = kwargs.get("temperature", 0.7)
        max_tokens = kwargs.get("max_tokens", 1024)

        if self.provider in ("openai", "openrouter"):
            stream = self._client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                stream=True,
            )
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield delta

        elif self.provider == "gemini":
            system_text, rest = self._split_system(messages)
            model = self._genai.GenerativeModel(
                self.model,
                system_instruction=system_text or None,
            )
            history = [
                {"role": ("user" if m["role"] == "user" else "model"),
                 "parts": [m["content"]]}
                for m in rest[:-1]
            ]
            chat = model.start_chat(history=history)
            stream = chat.send_message(
                rest[-1]["content"],
                generation_config={"temperature": temperature, "max_output_tokens": max_tokens},
                stream=True,
            )
            for chunk in stream:
                if chunk.text:
                    yield chunk.text

        elif self.provider == "claude":
            system_text, rest = self._split_system(messages)
            with self._client.messages.stream(
                model=self.model,
                max_tokens=max_tokens,
                temperature=temperature,
                system=system_text or None,
                messages=rest,
            ) as stream:
                for text in stream.text_stream:
                    yield text

### Código de prueba

Probamos el mismo prompt con todos los proveedores que tengas configurados.

In [ ]:
messages = [
    {"role": "system", "content": "Eres un asistente conciso. Responde en máximo 2 oraciones."},
    {"role": "user",   "content": "¿Qué es Python?"},
]

# Lista de proveedores según las keys disponibles
proveedores = ["openrouter"]
if OPENAI_API_KEY:    proveedores.append("openai")
if GOOGLE_API_KEY:    proveedores.append("gemini")
if ANTHROPIC_API_KEY: proveedores.append("claude")

for provider in proveedores:
    print(f"\n{'='*50}\nProveedor: {provider}\n{'='*50}")
    try:
        llm = LLMClient(provider)
        print(llm.chat(messages, temperature=0.7, max_tokens=200))
    except Exception as e:
        print(f"❌ Error: {e}")

### Prueba de streaming

In [ ]:
print(f"\n{'='*50}\nStreaming con OpenRouter\n{'='*50}")
llm = LLMClient("openrouter")
for token in llm.stream(messages, max_tokens=200):
    print(token, end="", flush=True)
print()

---

## 🎉 ¡Fin del notebook!

Has cubierto:

- ✅ Primera llamada a la API y lectura de metadatos
- ✅ Comparación entre modelos (OpenRouter + APIs nativas)
- ✅ Chatbot con gestión de memoria y resumen
- ✅ Extracción estructurada con validación y reintentos
- ✅ LangChain: `prompt | model | parser`
- ✅ Cliente unificado multi-proveedor

### Siguiente paso

La práctica evaluable de la sesión (Generador de Titulares) aplica todos estos patrones
a un caso real. Úsala para consolidar lo aprendido.